# ASF-UAV-Warning — демонстраційна Monte Carlo модель агентного мультисенсорного виявлення БпЛА
## Demonstrative Monte Carlo model of agentic multisensor UAV detection and public warning

Пакет відтворюваності до статті / Reproducibility package for the article:

> O. Korchenko, D. Prokopovych-Tkachenko, A. Desiatko, I. Azarov, O. Galushchenko, M. Mormul.
> **Agentic Multisensor System for Early Unmanned Aircraft Detection and Public Warning.**
> *Artificial Intelligence* (ISSN 2710-1673), 2026.

---

### 🔧 Архітектура цієї версії / Structure of this version

Код симуляції винесено з комірок ноутбука у **перевикористовуваний пакет `src/`**:

| Модуль | Відповідальність |
|---|---|
| `src/asf_simulation.py` | Генеративна модель: параметри сенсорів, генерація подій, оцінки трьох архітектур, латентності |
| `src/metrics.py` | Калібрування порогів, Таблиця 4 з bootstrap-CI, Pd за дальністю/умовами, абляція, часовий резерв |
| `src/make_figures.py` | Усі рисунки (inline, 320 dpi) — кожна функція повертає `matplotlib.Figure` |
| `tests/test_reproducibility.py` | `pytest`-тести: за `SEED = 20260` метрики Таблиці 4 збігаються з опублікованими **до 4 знаків** |

Ноутбук став **тонким оркестратором**: він лише викликає функції пакета й відображає
результати. Логіку тепер можна тестувати автономно (`pytest -q`) та перевикористовувати
поза Colab.

**Важливо / Important.** Усі числові результати є **демонстраційними**: вони характеризують
**синтетичну** генеративну модель, параметри якої задані авторами, а не реальні вимірювання.
Ноутбук призначений виключно для ілюстрації методології та перевірки відтворюваності обчислень.

**▶ Запуск у Colab:**
```
!git clone https://github.com/omega2417/bnt.git
%cd bnt/asf-uav-warning
```
далі *Runtime → Run all*. Використовуються лише `numpy`, `pandas`, `matplotlib`,
`scikit-learn`, які вже є в Colab.

## 0. Налаштування середовища / Environment setup

Підключаємо пакет `src/`, фіксуємо генератор випадкових чисел (`SEED = 20260`) — щоб
**кожен запуск давав ідентичні числа** — і створюємо робочі каталоги. Усі гіперпараметри
генеративної моделі задано в одному місці — `src/asf_simulation.py`.

In [ ]:
# ============================================================
# 0. Підключення пакета src/, каталоги, версії
# ============================================================
import os, sys, json, hashlib, warnings

def _find_project_root(start=None, depth=4):
    p = start or os.getcwd()
    for _ in range(depth):
        if os.path.isfile(os.path.join(p, "src", "asf_simulation.py")):
            return p
        p = os.path.dirname(p)
    return None

ROOT = _find_project_root() or os.getcwd()
sys.path.insert(0, ROOT)
os.chdir(ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src import asf_simulation as sim_mod
from src import metrics as met
from src import make_figures as figs
from src.asf_simulation import SEED, N_EVENTS, N_TEST, MODALITIES

warnings.filterwarnings("ignore")
figs.setup_matplotlib()

# Робочі каталоги (створюються в кореневій папці пакета)
for d in ("data", "results", "figures"):
    os.makedirs(d, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Seed: {SEED} | events: {N_EVENTS} | test: {N_TEST} | modalities: {MODALITIES}")
print("NumPy:", np.__version__, "| pandas:", pd.__version__)

## 1. Генерація синтетичного датасету / Synthetic dataset generation

Кожна подія $i$ описується:

* міткою класу $y_i \in \{0, 1\}$ — «фон» (птах, завада, цивільний ЛА) або «БпЛА»;
* дальністю $d_i \sim U(0{,}5;\; 8)$ км;
* умовами спостереження $c_i \in \{\text{clear, rain, fog, night, ew\_jam}\}$;
* оцінками впевненості чотирьох сенсорних агентів $s_{i,m} \in [0,1]$.

**Генеративна модель оцінки** модальності $m$:

$$s_{i,m} = \mathrm{clip}\Big( \mu_m(y_i, d_i, c_i) + \varepsilon_{i,m},\; 0,\; 1 \Big), \qquad \varepsilon_{i,m} \sim \mathcal{N}(0, \sigma_m^2)$$

де для БпЛА $\mu_m = \beta_m - \alpha_m d_i - \pi_m(c_i)$ (базовий рівень мінус деградація
з дальністю мінус штраф умов), а для фону $\mu_m = 0{,}22$ (константний шумовий рівень).

Крім того, кожен сенсор із імовірністю $p^{drop}_m$ **недоступний** (відмова, обрив каналу) —
це моделює реальну неповноту сенсорного поля.

| Модальність | Найчутливіша до | $p^{drop}_m$ |
|---|---|---|
| Радар | РЕБ-придушення (`ew_jam`) | 4 % |
| РЧ-аналізатор | РЕБ-придушення | 6 % |
| Акустика | Дощ (шум крапель) | 8 % |
| Оптика | Туман, ніч | 7 % |

Уся ця логіка — у `sim_mod.simulate()`. Він повертає об'єкт `SimulationData`, який
несе події, оцінки трьох архітектур, латентності **та живий генератор**, розташований
у точній позиції послідовності відтворюваності.

In [ ]:
# ============================================================
# 1. Синтетичний датасет (генеративна модель — у src/asf_simulation.py)
# ============================================================
sim = sim_mod.simulate(SEED)          # повний конвеєр: події → оцінки → латентності
df  = sim_mod.to_dataframe(sim)       # повний датафрейм подій
df.to_csv("data/synthetic_events.csv", index=False)

y, dist, cond = sim.y, sim.dist, sim.cond
print(f"Згенеровано {len(df):,} подій | БпЛА: {y.mean():.1%} | тест: {(df.split=='test').sum():,}")
df.head()

### 1.1. Огляд датасету / Dataset overview

Перевіряємо, що датасет виглядає «здоровим»: рівномірна дальність, задані частки умов,
і — головне — **розподіли оцінок сенсорів для БпЛА і фону перекриваються, але розділювані**.
Саме ступінь цього перекриття визначає досяжні Pd/FAR.

In [ ]:
# ============================================================
# 1.1. Візуальний огляд датасету (рис. 1)
# ============================================================
figs.fig_dataset_overview(sim, save="figures/fig1_dataset_overview.png"); plt.show()
figs.fig_optical_scores(sim); plt.show()

## 2. Три архітектури ухвалення рішення / Three decision architectures

1. **Один радарний агент (baseline).** Рішення лише за $s_{radar}$; якщо радар недоступний — оцінка 0.
2. **Статичне злиття.** Зважене середнє з **фіксованими** вагами $w = (0{,}35;\,0{,}30;\,0{,}15;\,0{,}20)$ по доступних сенсорах:
$$s^{stat}_i = \frac{\sum_m w_m a_{i,m} s_{i,m}}{\sum_m w_m a_{i,m}}$$
3. **Агентне злиття (запропонована архітектура).** Агент-координатор **адаптує ваги до контексту**: модальність, деградована поточними умовами, експоненційно послаблюється:
$$w^{ag}_{i,m} = w_m \exp\big(-\lambda\, \pi_m(c_i)\big), \qquad \lambda = 4$$
Так у туман система «довіряє» радару та РЧ, а під РЕБ — акустиці й оптиці.

Оцінки й латентності всіх трьох архітектур уже обчислено всередині `simulate()`
(функція `build_architectures`). **Латентність рішення** моделюється логнормальними
розподілами: агентний конвеєр ухвалює рішення раніше завдяки адаптивному гейтуванню
(медіана ≈ 0,85 с), статичне злиття чекає всі канали (≈ 1,56 с), одиночний радар — ≈ 1,23 с.

In [ ]:
# ============================================================
# 2. Оцінки та латентності трьох архітектур (уже в sim)
# ============================================================
for a in sim_mod.ARCH_NAMES:
    s = sim.arch[a]
    print(f"{sim_mod.ARCH_TITLES[a]:20s}: score∈[{s.min():.3f},{s.max():.3f}] "
          f"| медіана латентності ≈ {np.median(sim.latency[a]):.2f} с")

In [ ]:
# ============================================================
# 2.1. Візуалізація: розділюваність класів за архітектурами (рис. 2)
# ============================================================
figs.fig_score_separability(sim, save="figures/fig2_score_separability.png"); plt.show()

## 3. Метрики та довірчі інтервали / Metrics, thresholds, bootstrap CI

**Протокол.** Поріг спрацьовування $\tau$ кожної архітектури калібрується **на тренувальній
частині** за цільовим рівнем хибних тривог (квантиль фонових оцінок): 5,1 % / 4,3 % / 3,5 %.
Далі всі метрики обчислюються **лише на тестовій частині** (19 200 подій):

* **Pd** (probability of detection) — частка виявлених БпЛА;
* **FAR** (false alarm rate) — частка фонових подій, що спричинили тривогу;
* **Precision, F1, ROC AUC, Brier score**;
* медіана та P95 латентності.

95 % CI — **percentile bootstrap**, 800 перевибірок тестової множини. Уся ця логіка — у
`met.compute_metrics_table(sim)`.

In [ ]:
# ============================================================
# 3. Таблиця 4 статті: метрики + 95 % CI (bootstrap)
# ============================================================
metrics, thresholds = met.compute_metrics_table(sim)
metrics.to_csv("results/table4_metrics.csv", index=False)
metrics[["architecture","precision","pd","f1","far","roc_auc","brier",
         "latency_median_s","latency_p95_s"]].round(4)

### Таблиця 4 статті (демонстраційне відтворення)

Очікувані значення на тестовій частині (19 200 подій): **агентне злиття Pd ≈ 93 %, F1 ≈ 93–95 %,
FAR ≈ 3,5 %**; статичне злиття Pd ≈ 91–92 %, FAR ≈ 4,2 %; один радарний агент Pd ≈ 65–66 %,
FAR ≈ 5,1 %. Медіани затримки ≈ 0,85 / 1,56 / 1,23 с. Нижче — ті самі Pd/FAR із 95 % CI.

> Саме ці числа перевіряють `pytest`-тести (`tests/test_reproducibility.py`) — з точністю
> до 4 знаків після коми.

In [ ]:
# 95 % CI для Pd і FAR (транспоновано для компактності)
ci_cols = [c for c in metrics.columns if c.endswith(("_ci_lo", "_ci_hi"))]
display(metrics.set_index("architecture")[["pd", "far"] + ci_cols].round(4).T)

# --- Візуалізація Pd і FAR з довірчими інтервалами ---
figs.fig_metrics_bars(metrics, save="figures/fig_table4_bars.png"); plt.show()

## 4. Стійкість за дистанцією (рис. 3) / Pd vs distance

Розбиваємо тестові події-БпЛА на кілометрові інтервали дальності й оцінюємо Pd у кожному.
Очікування: перевага агентного злиття **зростає з дальністю**, бо саме на великих дистанціях
окремі канали деградують і виграш від адаптивного перезважування максимальний.

In [ ]:
# ============================================================
# 4. Pd за інтервалами дальності (рис. 3)
# ============================================================
dist_tab = met.pd_by_distance(sim, thresholds)
dist_tab.to_csv("results/pd_by_distance.csv", index=False)
figs.fig_pd_by_distance(dist_tab, save="figures/fig3_pd_by_distance.png"); plt.show()
dist_tab.pivot(index="dist_bin", columns="architecture", values="pd").round(4)

## 5. Компроміс FAR–Pd (рис. 4) / ROC and operating points

Повні ROC-криві на тестовій множині показують, що агентне злиття домінує на **всьому
діапазоні порогів**, а не лише в обраній робочій точці (позначено маркерами).

In [ ]:
# ============================================================
# 5. ROC-криві та робочі точки (рис. 4)
# ============================================================
figs.fig_roc(sim, metrics, save="figures/fig4_far_pd_tradeoff.png"); plt.show()

## 6. Метрики за умовами спостереження та абляційний аналіз (рис. 5)

**За умовами:** Pd кожної архітектури окремо для clear / rain / fog / night / ew_jam.
Найбільший розрив очікується під **РЕБ** (радар і РЧ придушені) та в **туман/ніч**
(оптика деградована).

**Абляція:** з агентного злиття по черзі вилучаємо одну модальність, повторно калібруємо
поріг (той самий цільовий FAR = 3,5 %) і вимірюємо F1. Просідання F1 показує **внесок
кожного сенсора** в загальну якість.

In [ ]:
# ============================================================
# 6a. Pd за умовами спостереження (рис. 5а)
# ============================================================
cond_tab = met.metrics_by_condition(sim, thresholds)
cond_tab.to_csv("results/metrics_by_condition.csv", index=False)
piv = cond_tab.pivot(index="cond", columns="architecture", values="pd").reindex(sim_mod.CONDS)
display(piv.round(4))
figs.fig_pd_by_condition(cond_tab, save="figures/fig5a_pd_by_condition.png"); plt.show()

In [ ]:
# ============================================================
# 6b. Абляція: агентне злиття без однієї модальності (рис. 5)
# ============================================================
abl = met.ablation(sim)
abl.to_csv("results/ablations.csv", index=False)
display(abl.round(4))
figs.fig_ablation(abl, save="figures/fig5_ablation_f1.png"); plt.show()

## 7. Латентність рішення (рис. 6) / Decision latency

Boxplot латентностей на тестовій частині. Агентний конвеєр ухвалює рішення раніше
завдяки адаптивному гейтуванню каналів: медіана ≈ 0,85 с проти 1,56 с у статичного злиття,
яке чекає завершення всіх каналів обробки.

In [ ]:
# ============================================================
# 7. Boxplot латентностей (рис. 6)
# ============================================================
figs.fig_latency_boxplot(sim, save="figures/fig6_latency_boxplot.png"); plt.show()

## 8. Часовий резерв після машинного рішення та людського підтвердження

Для кожної **виявленої** цілі (агентне злиття) оцінюємо часовий резерв до досягнення
об'єкта, що охороняється:

$$T^{arrival}_i = \frac{1000\, d_i}{v_i}, \quad v_i \sim U(15; 30)\ \text{м/с}$$

* резерв після машинного рішення: $M^{mach}_i = T^{arrival}_i - L_i$;
* резерв після людського підтвердження: $M^{hum}_i = M^{mach}_i - T^{conf}_i$,
  де $T^{conf}_i$ — логнормальний час підтвердження оператором (медіана ≈ 6 с).

Це відповідає вимозі протоколу «людина в контурі» перед запуском публічного оповіщення.

In [ ]:
# ============================================================
# 8. Часовий резерв (time margin)
# ============================================================
tm = met.time_margin(sim, thresholds)
tm.to_csv("results/time_margin.csv", index=False)

print("Медіана резерву після машинного рішення, с:", round(float(np.median(tm.margin_after_machine_s)), 2))
print("Медіана після людського підтвердження, с:  ", round(float(np.median(tm.margin_after_human_s)), 2))
print("P10 / P90 (після людини), с:",
      round(float(np.percentile(tm.margin_after_human_s, 10)), 2), "/",
      round(float(np.percentile(tm.margin_after_human_s, 90)), 2))

figs.fig_time_margin(tm, save="figures/fig7_time_margin.png"); plt.show()

## 9. Збережені артефакти та контрольні суми / Saved artifacts & checksums

Усі таблиці збережено в `data/` та `results/`, рисунки (320 dpi) — у `figures/`.
SHA-256 контрольні суми фіксують точний вміст файлів для депозиту (Zenodo тощо).
У Colab файли можна завантажити через панель *Files* ліворуч або `files.download(...)`.

In [ ]:
# ============================================================
# 9. Перелік артефактів + SHA-256
# ============================================================
manifest = {}
for root in ("data", "results", "figures"):
    for f in sorted(os.listdir(root)):
        p = os.path.join(root, f)
        manifest[p] = hashlib.sha256(open(p, "rb").read()).hexdigest()[:16]
with open("results/MANIFEST_sha256.json", "w") as fh:
    json.dump(manifest, fh, indent=2)
for p, h in manifest.items():
    print(f"{h}  {p}  ({os.path.getsize(p)/1024:.1f} КБ)")

## 10. Обмеження / Limitations

1. **Синтетичні дані:** розподіли оцінок, частки відмов і фонові класи задані авторами моделі, а не отримані з польових вимірювань.
2. **Подієва незалежність:** не моделюються часові кореляції, рої БпЛА, маскування, вплив рельєфу та багатопроменевість.
3. **Спрощений людський компонент:** час підтвердження — логнормальний розподіл без урахування втоми, стресу чи ергономіки інтерфейсу.
4. **Немає прямого емпіричного порівняння** з опублікованими системами виявлення.
5. Наведені числа **не можна використовувати** для закупівель, оцінки захищеності реальних об'єктів чи нормування часу евакуації.

### Відтворюваність / Reproducibility

```bash
cd asf-uav-warning
pip install -r requirements.txt
pytest -q                 # 29 тестів: метрики Таблиці 4 збігаються до 4 знаків
jupyter nbconvert --to notebook --execute ASF_UAV_Warning_Demo_Colab.ipynb
```

Ліцензія: MIT. Цитування — див. `CITATION.cff` у складі пакета відтворюваності.